# Pipeline de Extracción de Datos Comunes a cada Ciudad Española

Este *notebook* implementa un flujo completo de extracción, transformación y carga (ETL) para obtener un primer dataset que contenga características propias de cada ciudad española. Estas características son:

1. Latitud
2. Longitud
3. Altitud
4. Superficie
5. Distancia al mar
6. Factor cuenca

Estas variables se consiguen a partir de diversas técnicas de extracción: llamadas a APIs, realizando *Web Scraping* y a partir de cálculos matemáticos.

**Llamadas a APIs**

Una API (*Application Programming Interface*) es un conjunto de reglas que permite que dos aplicaciones se comuniquen entre sí. En este proyecto, nosotros actuamos como un "cliente" que solicita información específica a un "servidor" externo. En este script, enviamos una petición estructurada (normalmente una URL con parámetros como la latitud y longitud) y el servidor responde con los datos exactos en un formato ligero y fácil de leer para la máquina, generalmente JSON.

***Web Scraping***

El *Web Scraping* es una técnica que se utiliza cuando los datos que necesitamos están publicados en una página web pero no existe una API oficial para descargarlos de forma estructurada. En este script, leemos el código HTML de una página web (en este caso, Wikipedia) como si fuera un navegador, pero en lugar de mostrar imágenes y texto, buscamos etiquetas específicas donde está escondida la información. Utilizamos *BeautifulSoup* para navegar por el árbol de etiquetas HTML y *Requests* para descargar el contenido de la web.

In [ ]:
# =============================================================================
# LIBRERÍAS DE CIENCIA DE DATOS Y COMPUTACIÓN NUMÉRICA
# =============================================================================
import numpy as np           # Realiza cálculos matriciales y operaciones matemáticas vectorizadas.
import math                  # Proporciona funciones matemáticas fundamentales (seno, coseno, raíz cuadrada).
import pandas as pd          # Estructura los datos en DataFrames; esencial para limpieza y unión (merge) de archivos.

# =============================================================================
# LIBRERÍAS DE COMUNICACIÓN Y EXTRACCIÓN DE DATOS (SCRAPING)
# =============================================================================
import requests              # Gestiona las peticiones HTTP para descargar datos de Wikipedia y APIs externas.
from bs4 import BeautifulSoup # Analiza el código HTML de las webs para extraer datos específicos (como coordenadas).
import re                    # Expresiones Regulares: permite limpiar y extraer números de cadenas de texto complejas.

# =============================================================================
# LIBRERÍAS DE GESTIÓN DEL SISTEMA Y FLUJO DE TRABAJO
# =============================================================================
import os                    # Administra rutas de archivos y carpetas de forma compatible entre Windows/Linux.
import time                  # Controla los tiempos de ejecución y pausas (sleep) para evitar bloqueos de APIs.

# =============================================================================
# LIBRERÍAS DE ANÁLISIS GEOESPACIAL Y CARTOGRAFÍA (EL CORE TÉCNICO)
# =============================================================================

# Cartopy permite leer y manejar archivos de mapas profesionales (Shapefiles) de Natural Earth.
import cartopy.io.shapereader as shpreader 

# Shapely se encarga de la lógica geométrica:
# - Point: Representa la ciudad como un punto.
# - MultiPolygon: Maneja formas complejas como los océanos.
# - nearest_points: Encuentra la distancia mínima entre dos geometrías.
from shapely.geometry import Point, MultiPolygon
from shapely.ops import nearest_points, unary_union

# Geopy calcula la distancia geodésica real (en km) sobre la curvatura de la Tierra (elipsoide).
from geopy.distance import geodesic

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [2]:
# =============================================================================
# CONFIGURACIÓN E INICIALIZACIÓN DEL ENTORNO
# =============================================================================

# Definimos la ruta de salida.
# En esta carpeta se guardarán los archivos generados para cada ciudad.
RUTA_GUARDADO = os.path.join("..", "..", "datasets", "archivo_ciudades")

# Si la carpeta datasets\archivo_ciudades no existe, la creamos para evitar errores
# al guardar los resultados.
if not os.path.exists(RUTA_GUARDADO):
    print(f"📂 La carpeta no existe. Creándola en: {RUTA_GUARDADO}")
    os.makedirs(RUTA_GUARDADO)

# Este es un diccionario de las ciudades y sus nombres.
# Usamos el nombre como identificador único para futuros cruces de datos.
# El valor (nombre) es la clave de búsqueda exacta para generar la URL de Wikipedia.
ciudades = [
    "Madrid", "Barcelona", "Bilbao", "Sevilla", 
    "Valencia", "Valladolid", "Murcia", "Santa Cruz de Tenerife",
    "A Coruña", "Zaragoza", "Alicante/Alacant", "Albacete"
]

locations = {ciudad: ciudad for ciudad in ciudades}

📂 La carpeta no existe. Creándola en: ..\..\datasets\archivo_ciudades


In [3]:
# =============================================================================
# FUNCIONES DE EXTRACCIÓN (OPTIMIZADA SEGÚN TU CAPTURA DE PANTALLA)
# =============================================================================

def get_wiki_coordinates(nombre):
    """
    EXTRACCIÓN DE COORDENADAS MEDIANTE SCRAPING:
    Wikipedia almacena las coordenadas en varios formatos. El formato decimal es el más preciso
    para cálculos matemáticos posteriores.
    """
    # Construimos la URL dinámica sustituyendo espacios por guiones bajos
    url = f"https://es.wikipedia.org/wiki/{nombre.replace(' ', '_')}"

     # El 'User-Agent' es crucial: simula que somos un navegador real (Chrome/Firefox).
    # Sin esto, los servidores de Wikipedia podrían bloquear la petición al detectarla como un 'bot'.
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
    
    try:
        res = requests.get(url, headers=headers, timeout=5)
        # Parseamos el HTML con BeautifulSoup para poder navegar por el árbol de etiquetas
        soup = BeautifulSoup(res.text, 'html.parser')
        
        # Buscamos la etiqueta específica <span class="geo-dec"> que contiene la Lat/Lon decimal.
        # Este dato es inyectado por Wikipedia mediante plantillas geográficas.
        geo_container = soup.find('span', {'class': 'geo'})
        
        if geo_container:
            # El formato suele ser "lat; lon"
            texto_geo = geo_container.get_text().split(';')
            if len(texto_geo) >= 2:
                lat = float(texto_geo[0].strip())
                lon = float(texto_geo[1].strip())
                return round(lat, 5), round(lon, 5)

        # Si no, usamos el selector decimal estándar
        geo_span = soup.find('span', {'class': 'geo-dec'})
        if geo_span:
            texto_coords = geo_span.get_text()
            # Limpiamos posibles espacios o caracteres raros y buscamos el número
            nums = re.findall(r"[-+]?\d*\.\d+|\d+", texto_coords.replace(',', '.'))
            if len(nums) >= 2:
                return round(float(nums[0]), 5), round(float(nums[1]), 5)
                
    except Exception as e:
        print(f"Error en {nombre}: {e}")
    
    return None, None

In [4]:
def get_altitude(lat, lon):
    """
    CONEXIÓN CON API EXTERNA (OPEN-METEO):
    Cruza Lat/Lon con modelos de elevación digital (DEM) de alta resolución.
    Optimizado para evitar errores de conexión y asegurar datos numéricos limpios.
    """
    if lat is None or lon is None: 
        return None
        
    # Construimos la URL de la API de Open-Meteo
    url = f"https://api.open-meteo.com/v1/elevation?latitude={lat}&longitude={lon}"
    
    try:
        # Añadimos un timeout y verificamos el estado de la respuesta
        r = requests.get(url, timeout=10)
        r.raise_for_status() # Lanza error si el status no es 200
        
        datos = r.json()
        
        # Validamos que 'elevation' esté en el JSON y que sea una lista con contenido
        if 'elevation' in datos and len(datos['elevation']) > 0:
            altitud = datos['elevation'][0]
            
            # Devolvemos la altitud redondeada a 2 decimales para mayor limpieza
            return round(float(altitud), 2)
            
    except requests.exceptions.RequestException as e:
        print(f"Error de conexión al obtener altitud para ({lat}, {lon}): {e}")
    except (KeyError, IndexError, ValueError) as e:
        print(f"Error al procesar los datos de altitud: {e}")
        
    return None

In [5]:
def get_wiki_surface(nombre):
    """
    EXTRACCIÓN DE SUPERFICIE (INFOBOX SCRAPING):
    Este dato es fundamental para calcular el radio de influencia de cada lugar.
    Buscamos dentro de la tabla lateral (infobox) de la Wikipedia.
    """
    url = f"https://es.wikipedia.org/wiki/{nombre.replace(' ', '_')}"
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
    
    try:
        res = requests.get(url, headers=headers, timeout=5)
        soup = BeautifulSoup(res.text, 'html.parser')
        infobox = soup.find('table', {'class': 'infobox'})
        
        if not infobox: 
            return None
        
        # Iteramos por las filas de la tabla buscando las palabras clave
        for fila in infobox.find_all('tr'):
            texto_fila = fila.get_text().lower()
            if 'superficie' in texto_fila or 'área' in texto_fila:
                # El valor suele estar en la celda de datos (td). 
                # Si la fila solo tiene el título, buscamos en la siguiente fila.
                td = fila.find('td')
                if not td:
                    siguiente = fila.find_next_sibling('tr')
                    td = siguiente.find('td') if siguiente else None
                
                if td:
                    # Quitamos notas al pie (ej: [1], [25]) antes de procesar
                    for sup in td.find_all('sup'):
                        sup.decompose()
                        
                    raw_val = td.get_text().strip()
                    
                    # Limpieza de caracteres: quitamos km², espacios raros y puntos de miles
                    # El regex busca el primer bloque numérico que puede tener comas decimales
                    # Usamos una limpieza previa para quitar separadores de miles (puntos)
                    raw_val = raw_val.replace('\xa0', '').replace(' ', '')
                    
                    # Buscamos el patrón numérico: 1234,56 o 1234.56
                    match = re.search(r'(\d+[\.,]?\d*)', raw_val)
                    if match:
                        num_str = match.group(1)
                        
                        # Estandarización a formato float de Python (punto decimal)
                        # Si hay coma, es decimal (formato español habitual en Wiki)
                        if ',' in num_str:
                            num_str = num_str.replace('.', '').replace(',', '.')
                        
                        return float(num_str)
        return None
    except Exception as e:
        print(f"Error en {nombre}: {e}")
        return None

In [6]:
# =============================================================================
# ANÁLISIS GEOESPACIAL ((CÁLCULOS DERIVADOSPARA RESOLUCIÓN 10M)
# =============================================================================
from shapely.ops import nearest_points, unary_union

# Cargamos la costa con máxima resolución. 
# Al ser resolución 10m, el archivo contiene miles de LineStrings individuales.
try:
    coast_shp = shpreader.natural_earth(resolution='10m', category='physical', name='coastline')
    reader = shpreader.Reader(coast_shp)
    
    # Unificamos todas las líneas en una sola geometría para evitar errores de iteración
    coast_geom = unary_union(list(reader.geometries()))
    
    print("✓ Mapa de costa (10m) cargado y unificado con éxito.")
except Exception as e:
    print(f"× Error crítico al cargar shapefiles: {e}")

def get_distancia_mar(lat, lon):
    """
    CÁLCULO DE PROXIMIDAD COSTERA:
    Calcula la distancia mínima entre la ciudad y la línea de costa unificada.
    """
    if lat is None or lon is None: 
        return None
    
    # Creamos el objeto Punto (Longitud, Latitud)
    punto_ciudad = Point(lon, lat)
    
    try:
        # nearest_points encuentra el punto exacto de la costa más cercano a la ciudad
        _, punto_costa = nearest_points(punto_ciudad, coast_geom)
        
        # Geodesic calcula la distancia real en km considerando la curvatura terrestre
        distancia = geodesic((lat, lon), (punto_costa.y, punto_costa.x)).km
        
        return round(distancia, 2)
        
    except Exception as e:
        print(f"Error en cálculo geoespacial para ({lat}, {lon}): {e}")
        return None

✓ Mapa de costa (10m) cargado y unificado con éxito.


In [7]:
def get_factor_cuenca(lat, lon, alt_centro, superficie_km2):
    """
    ÍNDICE DE POSICIÓN TOPOGRÁFICA (TPI):
    Este cálculo determina si una ciudad está en un "agujero" (valle) o en una zona despejada.
    Las ciudades en cuencas (TPI negativo) suelen atrapar más la contaminación.
    Utiliza peticiones por lotes para evitar bloqueos y mejorar la velocidad.
    """
    if lat is None or lon is None or alt_centro is None: 
        return 0
    
    # ESTIMACIÓN DEL RADIO DE MUESTREO:
    # Si tenemos la superficie, calculamos el radio del círculo equivalente (A = π * r²)
    # Multiplicamos por 1.5 para analizar también las montañas que rodean la zona.
    radio_km = math.sqrt(superficie_km2 / math.pi) * 1.5 if superficie_km2 else 10

    # Convertimos KM a grados de arco terrestre (1 grado ≈ 111 km).
    radio_grados = radio_km / 111.0 

    elevaciones_entorno = []
    # MUESTREO CIRCULAR:
    # Generamos 50 puntos alrededor de la ciudad usando trigonometría (Seno y Coseno).
    # Esto crea un anillo de datos de elevación.
    for i in range(50):
        angulo = 2 * math.pi * i / 50
        # p_lat y p_lon definen un punto en el perímetro del círculo de búsqueda
        p_lat = lat + radio_grados * math.cos(angulo)
        p_lon = lon + radio_grados * math.sin(angulo)
        
        alt = get_altitude(p_lat, p_lon)
        if alt is not None: elevaciones_entorno.append(alt)
    
    if not elevaciones_entorno: return 0
    
    # LÓGICA DEL TPI:
    # TPI = Altitud de la ciudad - Media de las altitudes de alrededor.
    # Si el resultado es -100, significa que la ciudad está 100 metros por debajo de su entorno.
    promedio_entorno = sum(elevaciones_entorno) / len(elevaciones_entorno)
    return round(alt_centro - promedio_entorno, 2)
    
    

In [ ]:
# =============================================================================
# PROCESAMIENTO POR LOTES (MAIN LOOP)
# =============================================================================

lista_resultados = []

# Iteramos directamente sobre nuestra lista de ciudades españolas
for ciudad in ciudades:
    
    # PASO A: El scraping de coordenadas es la llave maestra. Sin esto no hay mapa.
    lat, lon = get_wiki_coordinates(ciudad)
    
    if lat is not None and lon is not None:
        # PASO B: Con las coordenadas obtenidas, disparamos el resto de peticiones.
        alt = get_altitude(lat, lon)
        sup = get_wiki_surface(ciudad)
        mar = get_distancia_mar(lat, lon)
        tpi = get_factor_cuenca(lat, lon, alt, sup)
        
        # Estructuramos el registro final
        lista_resultados.append({
            "Ciudad": ciudad,
            "Latitud": lat,
            "Longitud": lon,
            "Altitud (m)": alt,
            "Superficie (km²)": sup,
            "Distancia_Mar (km)": mar,
            "TPI_Factor_Cuenca": tpi
        })
    else:
        print(f" ❌ Error Crítico: No se hallaron coordenadas para {ciudad}. Saltando...")
    
    # CONTROL DE FLUJO:
    # Pausamos medio segundo entre peticiones para evitar que nos baneen por tráfico abusivo.
    time.sleep(0.5)

In [19]:
# =============================================================================
# EXPORTACIÓN Y CIERRE
# =============================================================================

# Creamos el objeto DataFrame, que es como una tabla de Excel en memoria de Python.
df = pd.DataFrame(lista_resultados)

# Guardamos el resultado en CSV con delimitador de punto y coma (;).
# Usamos 'utf-8-sig' para asegurar que Excel reconozca tildes y caracteres especiales.
nombre_archivo = "ciudades.csv"
df.to_csv(os.path.join(RUTA_GUARDADO, nombre_archivo), index=False, encoding='utf-8-sig', sep=';')

print(f"\n🏆 ¡PROCESO COMPLETADO!")
print(f"Se ha generado el archivo '{nombre_archivo}'.")


🏆 ¡PROCESO COMPLETADO!
Se ha generado el archivo 'ciudades.csv'.
